# EdgeAudioRecognition EfficientAT Fine-tuning

이 노트북은 Drive에 압축 푼 코드 폴더가 없어도 실행되도록 `/content/edgeaudio_efficientat_finetune_colab_fixed`에 필요한 `.py` 파일을 직접 생성합니다. 데이터셋만 Google Drive에 있으면 됩니다.

기본 흐름: Drive mount → 의존성 설치 → 학습 스크립트 생성 → `manifest.csv` 생성 → train/test split + K-fold fine-tuning → 예측 테스트.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, torch
print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git
!pip -q install pandas numpy scikit-learn soundfile librosa tqdm matplotlib


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os
LOCAL_CODE_DIR = '/content/edgeaudio_efficientat_finetune_colab_fixed'
os.makedirs(LOCAL_CODE_DIR, exist_ok=True)
%cd $LOCAL_CODE_DIR
print('LOCAL_CODE_DIR =', LOCAL_CODE_DIR)


/content/edgeaudio_efficientat_finetune_colab_fixed
LOCAL_CODE_DIR = /content/edgeaudio_efficientat_finetune_colab_fixed


In [ ]:
%%writefile prepare_manifest.py
#!/usr/bin/env python3
"""Build a single audio manifest from AI Hub CSV labels and ESC-50-style JSON sidecars.

Expected dataset layout example:

edge_audio_dataset/
├── dataset_labels.csv                  # AI Hub labels: file_path,label
├── 경보/*.wav
├── 물소리/*.wav
├── 비명/*.wav
├── 울음/*.wav
├── 유리깨지는소리/*.wav
├── 자전거/*.wav
├── 총/*.wav
├── 화재경보/*.wav
├── esc50/
│   ├── ESC50_5-231762-A-0.wav
│   └── ESC50_5-231762-A-0.json         # project_label, filename, fold, ...
└── ...

The output manifest has columns:
path,label,raw_label,source,group_id,fold,meta_path
"""
from __future__ import annotations

import argparse
import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd

AUDIO_EXTS_DEFAULT = (".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac")

# Project-level canonical labels. Edit this dict if you want Korean labels to remain Korean.
DEFAULT_LABEL_MAP: Dict[str, str] = {
    # AI Hub CSV labels in the uploaded example
    "경보": "alarm",
    "화재경보": "fire_alarm",
    "물소리": "water",
    "비명": "scream",
    "울음": "crying",
    "유리깨지는소리": "glass_break",
    "유리 깨지는 소리": "glass_break",
    "유리깨짐": "glass_break",
    "자전거": "bicycle",
    "총": "gunshot",
    "총성": "gunshot",
    "총소리": "gunshot",
    # ESC-50 / project JSON labels
    "babycry": "baby_cry",
    "baby_cry": "baby_cry",
    "crying_baby": "baby_cry",
    "crying baby": "baby_cry",
    "dog": "dog_bark",
    "dog_bark": "dog_bark",
    "dog bark": "dog_bark",
    "cat": "cat_meow",
    "cat_meow": "cat_meow",
    "cat meow": "cat_meow",
    "car horn": "car_horn",
    "car_horn": "car_horn",
    "carhorn": "car_horn",
    "door": "knock",
    "door_knock": "knock",
    "knock": "knock",
    "door_wood_knock": "knock"
}

PATH_COL_CANDIDATES = ("file_path", "path", "audio_path", "wav_path", "filename", "file", "audio")
LABEL_COL_CANDIDATES = ("project_label", "label", "class", "category", "source_category", "소분류", "중분류")
FOLD_COL_CANDIDATES = ("fold", "split", "esc50_fold")
SOURCE_COL_CANDIDATES = ("source", "source_dataset", "dataset")


def sanitize_label(label: Any) -> str:
    s = str(label).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return "unknown"
    s = s.replace("/", "_").replace("\\", "_").replace("-", "_")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9A-Za-z가-힣_]+", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s.lower() if re.search(r"[A-Za-z]", s) else s


def canonical_label(raw: Any, label_map: Dict[str, str]) -> str:
    s0 = str(raw).strip()
    s1 = sanitize_label(s0)
    candidates = [s0, s0.lower(), s1, s1.replace("_", " ")]
    for c in candidates:
        if c in label_map:
            return label_map[c]
    return s1


def load_label_map(path: Optional[Path]) -> Dict[str, str]:
    label_map = dict(DEFAULT_LABEL_MAP)
    if path is None:
        return label_map
    data = json.loads(path.read_text(encoding="utf-8"))
    for k, v in data.items():
        label_map[str(k)] = str(v)
    return label_map


def find_first_existing(candidates: Iterable[Path]) -> Optional[Path]:
    for p in candidates:
        if p.exists() and p.is_file():
            return p.resolve()
    return None


def build_basename_index(root: Path, exts: Sequence[str]) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = defaultdict(list)
    for ext in exts:
        for p in root.rglob(f"*{ext}"):
            index[p.name].append(p.resolve())
    return index


def _path_has_directory(raw_path: str) -> bool:
    """Return True when raw_path contains a directory component.

    CSV labels such as `경보/example.wav` should normally resolve only to
    `<data-root>/경보/example.wav`. Falling back to a basename-only match can
    silently attach the wrong label when another class has the same filename.
    """
    raw_norm = raw_path.replace("\\", "/")
    return "/" in raw_norm.strip("/")


def resolve_audio_path(
    raw_path: Any,
    root: Path,
    basename_index: Dict[str, List[Path]],
    base_dir: Optional[Path] = None,
    exts: Sequence[str] = AUDIO_EXTS_DEFAULT,
    allow_basename_fallback: bool = True,
) -> Optional[Path]:
    if raw_path is None:
        return None
    raw = str(raw_path).strip()
    if not raw or raw.lower() in {"nan", "none", "null"}:
        return None

    p = Path(raw)
    has_dir = _path_has_directory(raw)

    def add_exact_candidates(path_like: Path) -> List[Path]:
        cands: List[Path] = []
        if path_like.suffix.lower() in exts:
            cands.append(path_like)
        else:
            cands.append(path_like)
            for ext in exts:
                cands.append(path_like.with_suffix(ext))
                cands.append(Path(f"{path_like}{ext}"))
        return cands

    candidates: List[Path] = []
    if p.is_absolute():
        candidates.extend(add_exact_candidates(p))
    else:
        if base_dir is not None:
            candidates.extend(add_exact_candidates(base_dir / p))
        candidates.extend(add_exact_candidates(root / p))

        # Optional basename fallback. This is useful for JSON sidecars that only
        # contain a filename, but dangerous for CSV rows with class directories.
        if allow_basename_fallback or not has_dir:
            if base_dir is not None:
                candidates.extend(add_exact_candidates(base_dir / p.name))
            candidates.extend(add_exact_candidates(root / p.name))

    found = find_first_existing(candidates)
    if found is not None:
        return found

    # Last-resort basename lookup over the whole root. Keep this disabled for
    # directory-qualified CSV paths unless the user explicitly opts in.
    if not (allow_basename_fallback or not has_dir):
        return None

    names = [p.name]
    if p.suffix.lower() not in exts:
        names += [f"{p.name}{ext}" for ext in exts] + [p.with_suffix(ext).name for ext in exts]
    for name in names:
        matches = basename_index.get(name, [])
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            print(f"[WARN] basename lookup is ambiguous for {name}: {len(matches)} matches; using first")
            return matches[0]
    return None


def first_present(mapping: Dict[str, Any], keys: Sequence[str]) -> Optional[Any]:
    for k in keys:
        if k in mapping and mapping[k] not in (None, ""):
            return mapping[k]
    return None


def load_csv_rows(csv_path: Path, root: Path, label_map: Dict[str, str], basename_index: Dict[str, List[Path]], csv_basename_fallback: bool = False) -> List[Dict[str, Any]]:
    if not csv_path.exists():
        print(f"[CSV] not found: {csv_path}")
        return []

    df = pd.read_csv(csv_path)
    path_col = next((c for c in PATH_COL_CANDIDATES if c in df.columns), None)
    label_col = next((c for c in LABEL_COL_CANDIDATES if c in df.columns), None)
    fold_col = next((c for c in FOLD_COL_CANDIDATES if c in df.columns), None)
    source_col = next((c for c in SOURCE_COL_CANDIDATES if c in df.columns), None)

    if path_col is None or label_col is None:
        raise ValueError(
            f"CSV must contain an audio path column {PATH_COL_CANDIDATES} and label column {LABEL_COL_CANDIDATES}. "
            f"Found columns: {list(df.columns)}"
        )

    rows: List[Dict[str, Any]] = []
    missing = 0
    for _, r in df.iterrows():
        audio_path = resolve_audio_path(r[path_col], root, basename_index, allow_basename_fallback=csv_basename_fallback)
        if audio_path is None:
            missing += 1
            continue
        raw_label = str(r[label_col]).strip()
        rows.append(
            {
                "path": str(audio_path),
                "label": canonical_label(raw_label, label_map),
                "raw_label": raw_label,
                "source": str(r[source_col]) if source_col else "aihub_csv",
                "fold": str(r[fold_col]) if fold_col else "",
                "meta_path": str(csv_path.resolve()),
            }
        )
    print(f"[CSV] loaded={len(rows)} missing_audio={missing} from {csv_path}")
    return rows


def flatten_json_values(obj: Any) -> Iterable[str]:
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from flatten_json_values(v)
    elif isinstance(obj, list):
        for x in obj:
            yield from flatten_json_values(x)
    else:
        yield str(obj)


def load_json_rows(root: Path, label_map: Dict[str, str], basename_index: Dict[str, List[Path]], json_glob: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    json_paths = sorted(root.rglob(json_glob)) if "**" not in json_glob else sorted(root.glob(json_glob))
    if not json_paths:
        print(f"[JSON] no json matched: {root / json_glob}")
        return rows

    used_json = 0
    missing_audio = 0
    for jp in json_paths:
        try:
            try:
                data = json.loads(jp.read_text(encoding="utf-8"))
            except UnicodeDecodeError:
                data = json.loads(jp.read_text(encoding="cp949"))
        except Exception as e:
            print(f"[JSON] skip unreadable {jp}: {e}")
            continue

        if not isinstance(data, dict):
            continue

        raw_label = first_present(data, LABEL_COL_CANDIDATES)
        # Some AI Hub JSON versions hide the label deeper in nested fields. Use the whole JSON as fallback.
        if raw_label is None:
            raw_label = " ".join(flatten_json_values(data))

        raw_file = first_present(data, PATH_COL_CANDIDATES)
        audio_path = resolve_audio_path(raw_file, root, basename_index, base_dir=jp.parent) if raw_file else None
        if audio_path is None:
            # Sidecar convention: same stem as JSON.
            for ext in AUDIO_EXTS_DEFAULT:
                candidate = jp.with_suffix(ext)
                if candidate.exists():
                    audio_path = candidate.resolve()
                    break
        if audio_path is None:
            missing_audio += 1
            continue

        source = first_present(data, SOURCE_COL_CANDIDATES) or "json_sidecar"
        fold = first_present(data, FOLD_COL_CANDIDATES) or ""
        rows.append(
            {
                "path": str(audio_path),
                "label": canonical_label(raw_label, label_map),
                "raw_label": str(raw_label)[:300],
                "source": str(source),
                "fold": str(fold),
                "meta_path": str(jp.resolve()),
            }
        )
        used_json += 1

    print(f"[JSON] loaded={len(rows)} missing_audio={missing_audio} scanned={len(json_paths)}")
    return rows


def add_group_id(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    if mode == "none":
        df["group_id"] = [f"row_{i}" for i in range(len(df))]
    elif mode == "path":
        df["group_id"] = df["path"].astype(str)
    elif mode == "stem":
        df["group_id"] = df["path"].map(lambda x: Path(str(x)).stem)
    else:
        raise ValueError(f"Unknown group mode: {mode}")
    return df


def print_conflicts(df: pd.DataFrame) -> None:
    path_conf = df.groupby("path")["label"].nunique()
    path_conf = path_conf[path_conf > 1]
    stem_conf = df.assign(stem=df["path"].map(lambda x: Path(str(x)).stem)).groupby("stem")["label"].nunique()
    stem_conf = stem_conf[stem_conf > 1]
    if len(path_conf):
        print(f"[WARN] Same exact path has multiple labels: {len(path_conf)} paths")
        print(path_conf.head(10))
    if len(stem_conf):
        print(f"[WARN] Same basename stem has multiple labels: {len(stem_conf)} stems. Use --group-by stem to avoid train/test leakage.")
        print(stem_conf.head(10))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", type=Path, required=True, help="Dataset root containing CSV, audio folders, and JSON sidecars.")
    ap.add_argument("--csv", type=str, default="dataset_labels.csv", help="AI Hub CSV path. Relative paths are resolved under data-root.")
    ap.add_argument("--json-glob", type=str, default="**/*.json", help="JSON sidecar glob under data-root.")
    ap.add_argument("--output", type=Path, default=Path("manifest.csv"))
    ap.add_argument("--label-map-json", type=Path, default=None, help="Optional JSON mapping raw labels to canonical labels.")
    ap.add_argument("--group-by", choices=["stem", "path", "none"], default="stem")
    ap.add_argument("--audio-exts", nargs="+", default=list(AUDIO_EXTS_DEFAULT))
    ap.add_argument("--drop-missing", action="store_true", default=True)
    ap.add_argument("--csv-basename-fallback", action="store_true", help="Allow basename-only fallback for CSV file_path rows. Use only if your CSV paths are flattened and filenames are unique.")
    args = ap.parse_args()

    root = args.data_root.expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"data-root not found: {root}")

    exts = tuple(e if e.startswith(".") else f".{e}" for e in args.audio_exts)
    print(f"[SCAN] indexing audio under {root}")
    basename_index = build_basename_index(root, exts)
    print(f"[SCAN] indexed {sum(len(v) for v in basename_index.values())} audio files")

    label_map = load_label_map(args.label_map_json)

    csv_path = Path(args.csv)
    if not csv_path.is_absolute():
        csv_path = root / csv_path

    rows = []
    rows.extend(load_csv_rows(csv_path, root, label_map, basename_index, csv_basename_fallback=args.csv_basename_fallback))
    rows.extend(load_json_rows(root, label_map, basename_index, args.json_glob))

    if not rows:
        raise RuntimeError("No labeled audio found. Check --data-root, --csv, and JSON sidecars.")

    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["path", "label", "source", "meta_path"]).reset_index(drop=True)
    df = add_group_id(df, args.group_by)

    # Keep only rows whose file still exists.
    exists = df["path"].map(lambda x: Path(str(x)).exists())
    if not exists.all():
        print(f"[WARN] dropping {(~exists).sum()} rows whose files do not exist")
        df = df[exists].reset_index(drop=True)

    print_conflicts(df)

    args.output.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(args.output, index=False)
    print(f"\n[OK] manifest saved: {args.output.resolve()}")
    print("\n[label counts]")
    print(df["label"].value_counts())
    print("\n[source counts]")
    print(df["source"].value_counts())


if __name__ == "__main__":
    main()


Overwriting prepare_manifest.py


In [ ]:
%%writefile finetune_efficientat_kfold.py
#!/usr/bin/env python3
"""EfficientAT K-fold fine-tuning for the EdgeAudioRecognition project.

This script is intended for Colab/T4 or a desktop GPU, not Jetson Nano.
It performs:
1) fixed train/test split
2) K-fold cross-validation inside trainval
3) EfficientAT AudioSet-pretrained MobileNet head replacement
4) best-checkpoint selection by validation macro F1
5) final evaluation on the fixed test set for every fold
"""
from __future__ import annotations

import argparse
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import KFold, StratifiedGroupKFold, StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

try:
    import torchaudio
except Exception:
    torchaudio = None

try:
    import soundfile as sf
except Exception:
    sf = None

try:
    import librosa
except Exception:
    librosa = None


@dataclass
class TrainConfig:
    manifest: str
    out_dir: str
    efficientat_dir: str
    model_name: str = "mn04_as"
    head_type: str = "mlp"
    sample_rate: int = 32000
    clip_seconds: float = 10.0
    epochs: int = 20
    batch_size: int = 16
    lr: float = 1e-4
    weight_decay: float = 1e-4
    n_splits: int = 5
    test_size: float = 0.2
    seed: int = 42
    num_workers: int = 2
    patience: int = 6
    freeze_backbone_epochs: int = 2
    amp: bool = True
    weighted_sampler: bool = False
    class_weight: str = "balanced"
    max_samples_per_class: int = 0
    group_col: str = "group_id"
    label_col: str = "label"
    path_col: str = "path"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def run(cmd: Sequence[str], cwd: Optional[Path] = None) -> None:
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)


def safe_torch_load(path: Path, map_location: torch.device) -> Dict[str, Any]:
    """Load our own checkpoints across PyTorch versions.

    PyTorch 2.6 changed the default `weights_only` behavior. Our checkpoints
    contain only tensors and simple metadata, so explicitly disabling
    weights-only mode keeps loading stable across Colab runtime versions.
    """
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def ensure_efficientat_repo(repo_dir: Path) -> None:
    if not repo_dir.exists():
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "https://github.com/fschmid56/EfficientAT.git", repo_dir])
    if not (repo_dir / "models" / "mn" / "model.py").exists():
        raise FileNotFoundError(f"EfficientAT repo looks invalid: {repo_dir}")


def import_efficientat(repo_dir: Path):
    repo_dir = repo_dir.resolve()
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))
    # EfficientAT helpers load metadata/class_labels_indices.csv by relative path.
    os.chdir(repo_dir)
    from helpers.utils import NAME_TO_WIDTH  # type: ignore
    from models.mn.model import get_model as get_mn  # type: ignore
    from models.preprocess import AugmentMelSTFT  # type: ignore
    return NAME_TO_WIDTH, get_mn, AugmentMelSTFT


def load_audio(path: str, target_sr: int) -> torch.Tensor:
    """Return mono waveform as float tensor [T] at target_sr."""
    p = str(path)
    if torchaudio is not None:
        try:
            wav, sr = torchaudio.load(p)  # [C, T]
            wav = wav.float()
            if wav.ndim == 2 and wav.size(0) > 1:
                wav = wav.mean(dim=0, keepdim=True)
            if wav.ndim == 2:
                wav = wav.squeeze(0)
            if sr != target_sr:
                wav = torchaudio.functional.resample(wav, sr, target_sr)
            return wav.contiguous()
        except Exception:
            pass

    if sf is not None:
        data, sr = sf.read(p, always_2d=True, dtype="float32")
        wav_np = data.mean(axis=1)
        if sr != target_sr:
            if librosa is None:
                raise RuntimeError("Need librosa for resampling when torchaudio failed.")
            wav_np = librosa.resample(wav_np, orig_sr=sr, target_sr=target_sr)
        return torch.from_numpy(np.asarray(wav_np, dtype=np.float32)).contiguous()

    raise RuntimeError("No audio loader available. Install torchaudio or soundfile.")


class AudioClipDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        label2id: Dict[str, int],
        sample_rate: int = 32000,
        clip_seconds: float = 10.0,
        train: bool = False,
        path_col: str = "path",
        label_col: str = "label",
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.label2id = label2id
        self.sample_rate = int(sample_rate)
        self.clip_len = int(round(clip_seconds * sample_rate))
        self.train = train
        self.path_col = path_col
        self.label_col = label_col

    def __len__(self) -> int:
        return len(self.df)

    def _crop_or_pad(self, wav: torch.Tensor) -> torch.Tensor:
        n = wav.numel()
        if n <= 0:
            wav = torch.zeros(self.clip_len, dtype=torch.float32)
            n = wav.numel()
        if n < self.clip_len:
            pad = self.clip_len - n
            wav = torch.nn.functional.pad(wav, (0, pad))
        elif n > self.clip_len:
            if self.train:
                start = random.randint(0, n - self.clip_len)
            else:
                start = (n - self.clip_len) // 2
            wav = wav[start : start + self.clip_len]
        return wav

    def _augment_wave(self, wav: torch.Tensor) -> torch.Tensor:
        if not self.train:
            return wav
        # Random gain augmentation, safe for environmental audio.
        gain_db = random.uniform(-6.0, 6.0)
        wav = wav * float(10 ** (gain_db / 20.0))
        # Light Gaussian noise with low probability.
        if random.random() < 0.25:
            noise_std = 0.003 * torch.rand(1).item()
            wav = wav + torch.randn_like(wav) * noise_std
        return wav

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        row = self.df.iloc[idx]
        path = str(row[self.path_col])
        label = str(row[self.label_col])
        wav = load_audio(path, self.sample_rate)
        if wav.numel() > 0:
            wav = wav - wav.mean()
            peak = wav.abs().max().clamp_min(1e-5)
            wav = (wav / peak).clamp(-1.0, 1.0)
        wav = self._crop_or_pad(wav)
        wav = self._augment_wave(wav)
        y = torch.tensor(self.label2id[label], dtype=torch.long)
        return wav.float(), y, path


def collate_batch(batch: Sequence[Tuple[torch.Tensor, torch.Tensor, str]]) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    waves, labels, paths = zip(*batch)
    return torch.stack(list(waves), dim=0), torch.stack(list(labels), dim=0), list(paths)


def make_weighted_sampler(df: pd.DataFrame, label2id: Dict[str, int], label_col: str) -> WeightedRandomSampler:
    y = df[label_col].map(label2id).astype(int).to_numpy()
    counts = np.bincount(y, minlength=len(label2id)).astype(np.float64)
    counts[counts == 0] = 1.0
    weights_per_class = 1.0 / counts
    sample_weights = weights_per_class[y]
    return WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True)


def make_criterion(df_train: pd.DataFrame, label2id: Dict[str, int], label_col: str, mode: str, device: torch.device) -> nn.Module:
    if mode == "none":
        return nn.CrossEntropyLoss()
    y = df_train[label_col].map(label2id).astype(int).to_numpy()
    counts = np.bincount(y, minlength=len(label2id)).astype(np.float64)
    total = max(float(counts.sum()), 1.0)
    weights = total / (len(label2id) * np.maximum(counts, 1.0))
    # A fold may miss a very rare class. Give it a neutral weight instead of
    # letting sklearn.compute_class_weight crash. With enough data this should
    # not happen, but it keeps smoke tests from failing.
    weights[counts == 0] = 1.0
    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    print("[class counts]", {label: int(counts[idx]) for label, idx in label2id.items()})
    print("[class weights]", {label: float(weights_t[idx].cpu()) for label, idx in label2id.items()})
    return nn.CrossEntropyLoss(weight=weights_t)


def build_model(model_name: str, head_type: str, num_classes: int, device: torch.device, get_mn: Any, name_to_width: Any) -> nn.Module:
    model = get_mn(
        num_classes=num_classes,
        pretrained_name=model_name,
        width_mult=name_to_width(model_name),
        head_type=head_type,
    )
    model.to(device)
    return model


def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
    if hasattr(model, "features"):
        for p in model.features.parameters():
            p.requires_grad = trainable
    # Always keep classifier trainable.
    if hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True


def forward_logits(model: nn.Module, mel: nn.Module, waves: torch.Tensor) -> torch.Tensor:
    spec = mel(waves)  # [B, n_mels, frames]
    logits = model(spec.unsqueeze(1))
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    if logits.ndim > 2:
        logits = logits.flatten(start_dim=1)
    return logits


def train_one_epoch(
    model: nn.Module,
    mel: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
    device: torch.device,
    use_amp: bool,
) -> Dict[str, float]:
    model.train()
    mel.train()
    losses: List[float] = []
    ys: List[int] = []
    ps: List[int] = []
    pbar = tqdm(loader, desc="train", leave=False)
    for waves, y, _ in pbar:
        waves = waves.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp and device.type == "cuda"):
            logits = forward_logits(model, mel, waves)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
        pred = logits.detach().argmax(dim=1)
        ys.extend(y.detach().cpu().tolist())
        ps.extend(pred.cpu().tolist())
        pbar.set_postfix(loss=np.mean(losses[-20:]))
    return {
        "loss": float(np.mean(losses)) if losses else math.nan,
        "acc": float(accuracy_score(ys, ps)) if ys else math.nan,
        "macro_f1": float(f1_score(ys, ps, average="macro", zero_division=0)) if ys else math.nan,
    }


@torch.no_grad()
def evaluate(
    model: nn.Module,
    mel: nn.Module,
    loader: DataLoader,
    criterion: Optional[nn.Module],
    device: torch.device,
    use_amp: bool,
) -> Tuple[Dict[str, float], pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    mel.eval()
    losses: List[float] = []
    ys: List[int] = []
    ps: List[int] = []
    paths: List[str] = []
    probs_all: List[np.ndarray] = []
    for waves, y, batch_paths in tqdm(loader, desc="eval", leave=False):
        waves = waves.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with autocast(enabled=use_amp and device.type == "cuda"):
            logits = forward_logits(model, mel, waves)
            if criterion is not None:
                loss = criterion(logits, y)
                losses.append(float(loss.detach().cpu()))
        probs = torch.softmax(logits.float(), dim=1)
        pred = probs.argmax(dim=1)
        ys.extend(y.detach().cpu().tolist())
        ps.extend(pred.detach().cpu().tolist())
        paths.extend(batch_paths)
        probs_all.append(probs.detach().cpu().numpy())
    y_np = np.asarray(ys, dtype=np.int64)
    p_np = np.asarray(ps, dtype=np.int64)
    prob_np = np.concatenate(probs_all, axis=0) if probs_all else np.zeros((0, 0), dtype=np.float32)
    metrics = {
        "loss": float(np.mean(losses)) if losses else math.nan,
        "acc": float(accuracy_score(y_np, p_np)) if len(y_np) else math.nan,
        "macro_f1": float(f1_score(y_np, p_np, average="macro", zero_division=0)) if len(y_np) else math.nan,
        "weighted_f1": float(f1_score(y_np, p_np, average="weighted", zero_division=0)) if len(y_np) else math.nan,
    }
    pred_df = pd.DataFrame({"path": paths, "y_true": y_np, "y_pred": p_np})
    return metrics, pred_df, y_np, p_np, prob_np


def save_checkpoint(
    path: Path,
    model: nn.Module,
    cfg: TrainConfig,
    labels: List[str],
    fold: int,
    epoch: int,
    val_metrics: Dict[str, float],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": asdict(cfg),
            "labels": labels,
            "label2id": {label: i for i, label in enumerate(labels)},
            "id2label": {str(i): label for i, label in enumerate(labels)},
            "fold": fold,
            "epoch": epoch,
            "val_metrics": val_metrics,
        },
        path,
    )


def load_manifest(path: Path, cfg: TrainConfig) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in [cfg.path_col, cfg.label_col]:
        if col not in df.columns:
            raise ValueError(f"manifest missing required column '{col}'. columns={list(df.columns)}")
    df[cfg.path_col] = df[cfg.path_col].astype(str)
    df[cfg.label_col] = df[cfg.label_col].astype(str)
    exists = df[cfg.path_col].map(lambda x: Path(x).exists())
    if not exists.all():
        print(f"[WARN] dropping {(~exists).sum()} rows with missing audio files")
        df = df[exists].reset_index(drop=True)
    if cfg.max_samples_per_class and cfg.max_samples_per_class > 0:
        df = (
            df.groupby(cfg.label_col, group_keys=False)
            .apply(lambda g: g.sample(min(len(g), cfg.max_samples_per_class), random_state=cfg.seed))
            .reset_index(drop=True)
        )
    counts = df[cfg.label_col].value_counts()
    print("[manifest] rows:", len(df))
    print("[manifest] label counts:\n", counts)
    if len(counts) < 2:
        raise ValueError("Need at least 2 classes for classification.")
    return df


def make_labels(df: pd.DataFrame, label_col: str) -> List[str]:
    # Stable order: frequent labels first, then alphabetically for reproducibility.
    vc = df[label_col].value_counts()
    return sorted(vc.index.astype(str).tolist())


def reduce_splits_if_needed(df: pd.DataFrame, n_splits: int, label_col: str) -> int:
    min_count = int(df[label_col].value_counts().min())
    if min_count < 2:
        # Stratified K-fold is impossible. The caller will fall back to ordinary
        # KFold for debugging, but the result should not be used as final metric.
        new_n = min(max(2, n_splits), len(df))
        print(f"[WARN] minimum class count={min_count}. Stratified K-fold is impossible; fallback will use KFold(n_splits={new_n}).")
        return new_n
    if min_count < n_splits:
        new_n = max(2, min_count)
        print(f"[WARN] n_splits={n_splits} > minimum class count={min_count}. Reducing n_splits to {new_n}.")
        return new_n
    return n_splits


def train_test_split_group_safe(df: pd.DataFrame, cfg: TrainConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if cfg.test_size <= 0:
        return df.reset_index(drop=True), df.iloc[0:0].copy()

    y = df[cfg.label_col].astype(str).to_numpy()
    counts = pd.Series(y).value_counts()
    use_stratify = len(counts) > 1 and counts.min() >= 2

    # Prefer StratifiedGroupKFold for leakage prevention when group_id exists.
    if cfg.group_col in df.columns and use_stratify:
        n = max(2, int(round(1.0 / cfg.test_size)))
        n = min(n, int(counts.min()))
        if n >= 2:
            try:
                sgkf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=cfg.seed)
                groups = df[cfg.group_col].astype(str).to_numpy()
                train_idx, test_idx = next(sgkf.split(df, y, groups))
                return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)
            except Exception as e:
                print(f"[WARN] StratifiedGroupKFold test split failed, fallback to train_test_split: {e}")

    stratify = y if use_stratify else None
    train_df, test_df = train_test_split(df, test_size=cfg.test_size, random_state=cfg.seed, stratify=stratify)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def iter_kfolds(df: pd.DataFrame, cfg: TrainConfig) -> Iterable[Tuple[int, np.ndarray, np.ndarray]]:
    y = df[cfg.label_col].astype(str).to_numpy()
    n_splits = reduce_splits_if_needed(df, cfg.n_splits, cfg.label_col)
    min_count = int(pd.Series(y).value_counts().min())

    if min_count < 2:
        print("[WARN] Using ordinary KFold because at least one class has fewer than 2 samples. Use full dataset for meaningful K-fold metrics.")
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=cfg.seed)
        for fold, (tr, va) in enumerate(splitter.split(df), start=1):
            yield fold, tr, va
        return

    if cfg.group_col in df.columns:
        groups = df[cfg.group_col].astype(str).to_numpy()
        try:
            splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.seed)
            for fold, (tr, va) in enumerate(splitter.split(df, y, groups), start=1):
                yield fold, tr, va
            return
        except Exception as e:
            print(f"[WARN] StratifiedGroupKFold failed, fallback to StratifiedKFold: {e}")

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.seed)
    for fold, (tr, va) in enumerate(splitter.split(df, y), start=1):
        yield fold, tr, va


def make_loader(
    df: pd.DataFrame,
    label2id: Dict[str, int],
    cfg: TrainConfig,
    train: bool,
    sampler: Optional[WeightedRandomSampler] = None,
) -> DataLoader:
    ds = AudioClipDataset(
        df=df,
        label2id=label2id,
        sample_rate=cfg.sample_rate,
        clip_seconds=cfg.clip_seconds,
        train=train,
        path_col=cfg.path_col,
        label_col=cfg.label_col,
    )
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=(train and sampler is None),
        sampler=sampler,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        collate_fn=collate_batch,
    )


def write_report(
    out_dir: Path,
    prefix: str,
    labels: List[str],
    y_true: np.ndarray,
    y_pred: np.ndarray,
    pred_df: pd.DataFrame,
    prob_np: np.ndarray,
) -> Dict[str, Any]:
    report = classification_report(y_true, y_pred, labels=np.arange(len(labels)), target_names=labels, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    pred_out = pred_df.copy()
    pred_out["true_label"] = pred_out["y_true"].map(lambda i: labels[int(i)])
    pred_out["pred_label"] = pred_out["y_pred"].map(lambda i: labels[int(i)])
    if prob_np.size:
        for i, label in enumerate(labels):
            pred_out[f"prob_{label}"] = prob_np[:, i]
    pred_out.to_csv(out_dir / f"{prefix}_predictions.csv", index=False)
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(out_dir / f"{prefix}_confusion_matrix.csv")
    (out_dir / f"{prefix}_classification_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"classification_report": report, "confusion_matrix": cm.tolist()}


def run_fold(
    fold: int,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cfg: TrainConfig,
    labels: List[str],
    device: torch.device,
    get_mn: Any,
    name_to_width: Any,
    AugmentMelSTFT: Any,
) -> Dict[str, Any]:
    label2id = {label: i for i, label in enumerate(labels)}
    fold_dir = Path(cfg.out_dir) / f"fold_{fold:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    split_df = pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="val"),
            test_df.assign(split="test"),
        ],
        ignore_index=True,
    )
    split_df.to_csv(fold_dir / "split.csv", index=False)

    sampler = make_weighted_sampler(train_df, label2id, cfg.label_col) if cfg.weighted_sampler else None
    train_loader = make_loader(train_df, label2id, cfg, train=True, sampler=sampler)
    val_loader = make_loader(val_df, label2id, cfg, train=False)
    test_loader = make_loader(test_df, label2id, cfg, train=False) if len(test_df) else None

    model = build_model(cfg.model_name, cfg.head_type, len(labels), device, get_mn, name_to_width)
    mel = AugmentMelSTFT(n_mels=128, sr=cfg.sample_rate, win_length=800, hopsize=320, freqm=48, timem=192)
    mel.to(device)

    criterion = make_criterion(train_df, label2id, cfg.label_col, cfg.class_weight, device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs))
    scaler = GradScaler(enabled=cfg.amp and device.type == "cuda")

    history: List[Dict[str, Any]] = []
    best_macro_f1 = -1.0
    best_epoch = -1
    best_path = fold_dir / "best.pt"
    bad_epochs = 0

    for epoch in range(1, cfg.epochs + 1):
        if cfg.freeze_backbone_epochs > 0:
            freeze = epoch <= cfg.freeze_backbone_epochs
            set_backbone_trainable(model, trainable=not freeze)
            # Rebuild optimizer exactly when unfreezing starts.
            if epoch == cfg.freeze_backbone_epochs + 1:
                optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs - epoch + 1))
        t0 = time.time()
        train_metrics = train_one_epoch(model, mel, train_loader, optimizer, criterion, scaler, device, cfg.amp)
        val_metrics, val_pred_df, yv, pv, probv = evaluate(model, mel, val_loader, criterion, device, cfg.amp)
        scheduler.step()
        row = {
            "fold": fold,
            "epoch": epoch,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "seconds": float(time.time() - t0),
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        pd.DataFrame(history).to_csv(fold_dir / "history.csv", index=False)
        print(
            f"[fold {fold} epoch {epoch:03d}] "
            f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['macro_f1']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['macro_f1']:.4f} val_acc={val_metrics['acc']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            bad_epochs = 0
            save_checkpoint(best_path, model, cfg, labels, fold, epoch, val_metrics)
            write_report(fold_dir, "val_best_current", labels, yv, pv, val_pred_df, probv)
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.patience:
                print(f"[fold {fold}] early stopping at epoch={epoch}, best_epoch={best_epoch}")
                break

    # Reload best checkpoint for final fold evaluation.
    ckpt = safe_torch_load(best_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    val_metrics, val_pred_df, yv, pv, probv = evaluate(model, mel, val_loader, criterion, device, cfg.amp)
    val_report = write_report(fold_dir, "val", labels, yv, pv, val_pred_df, probv)

    test_metrics: Dict[str, float] = {}
    test_report: Dict[str, Any] = {}
    if test_loader is not None:
        test_metrics, test_pred_df, yt, pt, probt = evaluate(model, mel, test_loader, criterion, device, cfg.amp)
        test_report = write_report(fold_dir, "test", labels, yt, pt, test_pred_df, probt)

    metrics = {
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_macro_f1_seen": best_macro_f1,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "val_report": val_report,
        "test_report": test_report,
        "checkpoint": str(best_path),
    }
    (fold_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    return metrics


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", type=Path, required=True)
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--efficientat-dir", type=Path, default=Path("/content/EfficientAT"))
    ap.add_argument("--model-name", default="mn04_as", choices=["mn04_as", "mn05_as", "mn10_as", "mn10_as_hop_20", "mn10_as_mels_64"])
    ap.add_argument("--head-type", default="mlp", choices=["mlp", "fully_convolutional"])
    ap.add_argument("--sample-rate", type=int, default=32000)
    ap.add_argument("--clip-seconds", type=float, default=10.0)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--weight-decay", type=float, default=1e-4)
    ap.add_argument("--n-splits", type=int, default=5)
    ap.add_argument("--test-size", type=float, default=0.2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--num-workers", type=int, default=2)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--freeze-backbone-epochs", type=int, default=2)
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--weighted-sampler", action="store_true")
    ap.add_argument("--class-weight", choices=["balanced", "none"], default="balanced")
    ap.add_argument("--max-samples-per-class", type=int, default=0, help="0 means no cap; useful for quick debug.")
    ap.add_argument("--group-col", default="group_id")
    ap.add_argument("--label-col", default="label")
    ap.add_argument("--path-col", default="path")
    args = ap.parse_args()

    cfg = TrainConfig(
        manifest=str(args.manifest),
        out_dir=str(args.out_dir),
        efficientat_dir=str(args.efficientat_dir),
        model_name=args.model_name,
        head_type=args.head_type,
        sample_rate=args.sample_rate,
        clip_seconds=args.clip_seconds,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        weight_decay=args.weight_decay,
        n_splits=args.n_splits,
        test_size=args.test_size,
        seed=args.seed,
        num_workers=args.num_workers,
        patience=args.patience,
        freeze_backbone_epochs=args.freeze_backbone_epochs,
        amp=not args.no_amp,
        weighted_sampler=args.weighted_sampler,
        class_weight=args.class_weight,
        max_samples_per_class=args.max_samples_per_class,
        group_col=args.group_col,
        label_col=args.label_col,
        path_col=args.path_col,
    )

    seed_everything(cfg.seed)
    cfg_out = Path(cfg.out_dir)
    cfg_out.mkdir(parents=True, exist_ok=True)
    (cfg_out / "config.json").write_text(json.dumps(asdict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")

    ensure_efficientat_repo(Path(cfg.efficientat_dir))
    name_to_width, get_mn, AugmentMelSTFT = import_efficientat(Path(cfg.efficientat_dir))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("[device]", device)
    if device.type == "cuda":
        print("[gpu]", torch.cuda.get_device_name(0))

    df = load_manifest(Path(cfg.manifest), cfg)
    labels = make_labels(df, cfg.label_col)
    label2id = {label: i for i, label in enumerate(labels)}
    (cfg_out / "labels.json").write_text(json.dumps({"labels": labels, "label2id": label2id}, ensure_ascii=False, indent=2), encoding="utf-8")

    trainval_df, test_df = train_test_split_group_safe(df, cfg)
    print(f"[split] trainval={len(trainval_df)} test={len(test_df)}")
    trainval_df.to_csv(cfg_out / "trainval.csv", index=False)
    test_df.to_csv(cfg_out / "test.csv", index=False)

    fold_metrics: List[Dict[str, Any]] = []
    best_fold_metric = -1.0
    best_ckpt_path: Optional[Path] = None

    for fold, tr_idx, va_idx in iter_kfolds(trainval_df, cfg):
        train_df = trainval_df.iloc[tr_idx].reset_index(drop=True)
        val_df = trainval_df.iloc[va_idx].reset_index(drop=True)
        print(f"\n========== FOLD {fold} ==========")
        print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")
        print("train label counts:\n", train_df[cfg.label_col].value_counts())
        print("val label counts:\n", val_df[cfg.label_col].value_counts())
        metrics = run_fold(
            fold,
            train_df,
            val_df,
            test_df,
            cfg,
            labels,
            device,
            get_mn,
            name_to_width,
            AugmentMelSTFT,
        )
        fold_metrics.append(metrics)
        current = float(metrics["val_metrics"].get("macro_f1", -1.0))
        if current > best_fold_metric:
            best_fold_metric = current
            best_ckpt_path = Path(metrics["checkpoint"])

    # Save compact summary.
    rows = []
    for m in fold_metrics:
        row = {"fold": m["fold"], "best_epoch": m["best_epoch"], "checkpoint": m["checkpoint"]}
        row.update({f"val_{k}": v for k, v in m["val_metrics"].items()})
        row.update({f"test_{k}": v for k, v in m["test_metrics"].items()})
        rows.append(row)
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(cfg_out / "kfold_summary.csv", index=False)
    (cfg_out / "all_metrics.json").write_text(json.dumps(fold_metrics, ensure_ascii=False, indent=2), encoding="utf-8")

    if best_ckpt_path and best_ckpt_path.exists():
        shutil.copy2(best_ckpt_path, cfg_out / "best_model.pt")
        print(f"[OK] best checkpoint copied: {cfg_out / 'best_model.pt'}")
    print(f"[OK] summary saved: {cfg_out / 'kfold_summary.csv'}")


if __name__ == "__main__":
    main()


Overwriting finetune_efficientat_kfold.py


In [ ]:
%%writefile predict_efficientat.py
#!/usr/bin/env python3
"""Run inference with a fine-tuned EfficientAT checkpoint.

Example:
python predict_efficientat.py \
  --checkpoint /content/drive/MyDrive/edge_audio_runs/effat_mn04/best_model.pt \
  --audio /content/drive/MyDrive/edge_audio_dataset/example.wav \
  --topk 5 --chunk-seconds 10 --hop-seconds 5
"""
from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import numpy as np
import torch
from torch.cuda.amp import autocast

try:
    import torchaudio
except Exception:
    torchaudio = None
try:
    import soundfile as sf
except Exception:
    sf = None
try:
    import librosa
except Exception:
    librosa = None


def run(cmd: Sequence[str]) -> None:
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)


def safe_torch_load(path: Path, map_location: torch.device) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def ensure_efficientat_repo(repo_dir: Path) -> None:
    if not repo_dir.exists():
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "https://github.com/fschmid56/EfficientAT.git", repo_dir])
    if str(repo_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(repo_dir.resolve()))
    os.chdir(repo_dir.resolve())


def import_efficientat(repo_dir: Path):
    ensure_efficientat_repo(repo_dir)
    from helpers.utils import NAME_TO_WIDTH  # type: ignore
    from models.mn.model import get_model as get_mn  # type: ignore
    from models.preprocess import AugmentMelSTFT  # type: ignore
    return NAME_TO_WIDTH, get_mn, AugmentMelSTFT


def load_audio(path: str, target_sr: int) -> torch.Tensor:
    if torchaudio is not None:
        try:
            wav, sr = torchaudio.load(path)
            wav = wav.float()
            if wav.ndim == 2 and wav.size(0) > 1:
                wav = wav.mean(dim=0, keepdim=True)
            if wav.ndim == 2:
                wav = wav.squeeze(0)
            if sr != target_sr:
                wav = torchaudio.functional.resample(wav, sr, target_sr)
            return wav.contiguous()
        except Exception:
            pass
    if sf is not None:
        data, sr = sf.read(path, always_2d=True, dtype="float32")
        wav_np = data.mean(axis=1)
        if sr != target_sr:
            if librosa is None:
                raise RuntimeError("Need librosa for resampling when torchaudio failed.")
            wav_np = librosa.resample(wav_np, orig_sr=sr, target_sr=target_sr)
        return torch.from_numpy(np.asarray(wav_np, dtype=np.float32)).contiguous()
    raise RuntimeError("Install torchaudio or soundfile.")


def make_chunks(wav: torch.Tensor, sample_rate: int, chunk_seconds: float, hop_seconds: float) -> torch.Tensor:
    chunk = int(round(chunk_seconds * sample_rate))
    hop = int(round(hop_seconds * sample_rate))
    if wav.numel() == 0:
        wav = torch.zeros(chunk)
    wav = wav - wav.mean()
    wav = wav / wav.abs().max().clamp_min(1e-5)
    if wav.numel() <= chunk:
        return torch.nn.functional.pad(wav, (0, chunk - wav.numel())).unsqueeze(0)
    chunks = []
    for start in range(0, wav.numel() - chunk + 1, hop):
        chunks.append(wav[start : start + chunk])
    if not chunks or chunks[-1].numel() < chunk:
        chunks.append(wav[-chunk:])
    return torch.stack(chunks, dim=0)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", type=Path, required=True)
    ap.add_argument("--audio", type=Path, required=True)
    ap.add_argument("--efficientat-dir", type=Path, default=Path("/content/EfficientAT"))
    ap.add_argument("--topk", type=int, default=5)
    ap.add_argument("--chunk-seconds", type=float, default=10.0)
    ap.add_argument("--hop-seconds", type=float, default=5.0)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    device = torch.device(args.device)
    ckpt = safe_torch_load(args.checkpoint, map_location=device)
    labels: List[str] = ckpt["labels"]
    cfg: Dict[str, Any] = ckpt.get("config", {})
    model_name = cfg.get("model_name", "mn04_as")
    head_type = cfg.get("head_type", "mlp")
    sample_rate = int(cfg.get("sample_rate", 32000))

    NAME_TO_WIDTH, get_mn, AugmentMelSTFT = import_efficientat(args.efficientat_dir)
    model = get_mn(num_classes=len(labels), pretrained_name=None, width_mult=NAME_TO_WIDTH(model_name), head_type=head_type)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.to(device).eval()
    mel = AugmentMelSTFT(n_mels=128, sr=sample_rate, win_length=800, hopsize=320, freqm=0, timem=0)
    mel.to(device).eval()

    wav = load_audio(str(args.audio), sample_rate)
    chunks = make_chunks(wav, sample_rate, args.chunk_seconds, args.hop_seconds).to(device)

    probs_all = []
    with torch.no_grad(), autocast(enabled=device.type == "cuda"):
        for i in range(0, chunks.size(0), 16):
            batch = chunks[i : i + 16]
            spec = mel(batch)
            out = model(spec.unsqueeze(1))
            logits = out[0] if isinstance(out, (tuple, list)) else out
            probs_all.append(torch.softmax(logits.float(), dim=1))
    probs = torch.cat(probs_all, dim=0).mean(dim=0).detach().cpu().numpy()
    order = probs.argsort()[::-1][: args.topk]
    result = {
        "audio": str(args.audio),
        "checkpoint": str(args.checkpoint),
        "num_chunks": int(chunks.size(0)),
        "topk": [{"label": labels[int(i)], "score": float(probs[int(i)])} for i in order],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


Overwriting predict_efficientat.py


In [ ]:
!python -m py_compile prepare_manifest.py finetune_efficientat_kfold.py predict_efficientat.py
!ls -lah


total 68K
drwxr-xr-x 3 root root 4.0K May 25 12:56 .
drwxr-xr-x 1 root root 4.0K May 25 12:57 ..
-rw-r--r-- 1 root root  31K May 25 13:36 finetune_efficientat_kfold.py
-rw-r--r-- 1 root root 5.8K May 25 13:36 predict_efficientat.py
-rw-r--r-- 1 root root  15K May 25 13:36 prepare_manifest.py
drwxr-xr-x 2 root root 4.0K May 25 13:36 __pycache__


## 경로 설정

`DATA_ROOT`는 전체 데이터셋 폴더입니다. 폴더 안에 `dataset_labels.csv`가 있고, CSV의 `file_path`가 예를 들어 `경보/example.wav`라면 실제 파일도 `DATA_ROOT/경보/example.wav`에 있어야 합니다.


In [ ]:
DATA_ROOT = '/content/drive/MyDrive/edge_audio_dataset'
RUN_ROOT = '/content/drive/MyDrive/edge_audio_runs/effat_mn04_kfold'

import os
os.makedirs(RUN_ROOT, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('RUN_ROOT  =', RUN_ROOT)
print('DATA_ROOT exists:', os.path.exists(DATA_ROOT))
print('CSV exists:', os.path.exists(os.path.join(DATA_ROOT, 'dataset_labels.csv')))

!find "$DATA_ROOT" -maxdepth 2 -type f \( -name "*.wav" -o -name "*.json" -o -name "dataset_labels.csv" \) | head -30


DATA_ROOT = /content/drive/MyDrive/edge_audio_dataset
RUN_ROOT  = /content/drive/MyDrive/edge_audio_runs/effat_mn04_kfold
DATA_ROOT exists: True
CSV exists: True
/content/drive/MyDrive/edge_audio_dataset/dataset_labels.csv
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_205_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_208_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_214_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_212_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_211_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_222_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_210_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_209_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_215_0001.wav
/content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_228_0001.wav
/content/drive/My

In [ ]:
!python prepare_manifest.py   --data-root "$DATA_ROOT"   --csv dataset_labels.csv   --json-glob "**/*.json"   --group-by stem   --output "$RUN_ROOT/manifest.csv"


[SCAN] indexing audio under /content/drive/MyDrive/edge_audio_dataset
[SCAN] indexed 3491 audio files
[CSV] loaded=3291 missing_audio=0 from /content/drive/MyDrive/edge_audio_dataset/dataset_labels.csv
[JSON] loaded=200 missing_audio=0 scanned=200
[WARN] Same basename stem has multiple labels: 171 stems. Use --group-by stem to avoid train/test leakage.
stem
S-211014_S_103_C_125_0001    2
S-211014_S_103_C_126_0001    2
S-211014_S_103_C_127_0001    2
S-211014_S_103_C_128_0001    2
S-211014_S_103_C_129_0001    2
S-211014_S_103_C_130_0001    2
S-211014_S_103_C_131_0001    2
S-211014_S_103_C_132_0001    2
S-211014_S_103_C_133_0001    2
S-211014_S_103_C_134_0001    2
Name: label, dtype: int64

[OK] manifest saved: /content/drive/MyDrive/edge_audio_runs/effat_mn04_kfold/manifest.csv

[label counts]
label
gunshot        1670
glass_break     482
alarm           409
water           238
fire_alarm      171
crying          146
scream           94
bicycle          81
dog_bark         40
knock      

In [ ]:
import pandas as pd, os
manifest_path = f'{RUN_ROOT}/manifest.csv'
manifest = pd.read_csv(manifest_path)
print('manifest rows:', len(manifest))
display(manifest.head())
display(manifest['label'].value_counts())


manifest rows: 3491


,path,label,raw_label,source,fold,meta_path,group_id
0,/content/drive/MyDrive/edge_audio_dataset/경보/S...,alarm,경보,aihub_csv,NaN,/content/drive/MyDrive/edge_audio_dataset/data...,S-210915_S_101_D_002_0001
1,/content/drive/MyDrive/edge_audio_dataset/경보/S...,alarm,경보,aihub_csv,NaN,/content/drive/MyDrive/edge_audio_dataset/data...,S-211007_S_101_D_001_0001
2,/content/drive/MyDrive/edge_audio_dataset/경보/S...,alarm,경보,aihub_csv,NaN,/content/drive/MyDrive/edge_audio_dataset/data...,S-211014_S_101_C_003_0001
3,/content/drive/MyDrive/edge_audio_dataset/경보/S...,alarm,경보,aihub_csv,NaN,/content/drive/MyDrive/edge_audio_dataset/data...,S-211014_S_101_C_004_0001
4,/content/drive/MyDrive/edge_audio_dataset/경보/S...,alarm,경보,aihub_csv,NaN,/content/drive/MyDrive/edge_audio_dataset/data...,S-211014_S_103_C_125_0001


,count
label,
gunshot,1670
glass_break,482
alarm,409
water,238
fire_alarm,171
crying,146
scream,94
bicycle,81
dog_bark,40


## EfficientAT K-fold fine-tuning

처음에는 `QUICK_DEBUG = True`로 1~2 epoch만 돌려서 경로와 코드가 정상인지 확인 후 `False`로 바꿔 전체 학습 진행


In [ ]:
QUICK_DEBUG = False # 경로/코드 확인 후 전체 학습할 때 False로 변경

if QUICK_DEBUG:
    EXTRA_ARGS = '--epochs 2 --batch-size 8 --n-splits 2 --max-samples-per-class 20 --patience 2'
else:
    EXTRA_ARGS = '--epochs 20 --batch-size 16 --n-splits 5 --patience 6'

print('EXTRA_ARGS =', EXTRA_ARGS)


EXTRA_ARGS = --epochs 20 --batch-size 16 --n-splits 5 --patience 6


In [ ]:
!python finetune_efficientat_kfold.py   --manifest "$RUN_ROOT/manifest.csv"   --out-dir "$RUN_ROOT"   --efficientat-dir /content/EfficientAT   --model-name mn04_as   --test-size 0.2   --freeze-backbone-epochs 2   --weighted-sampler   --num-workers 0   $EXTRA_ARGS


[device] cuda
[gpu] Tesla T4
[manifest] rows: 3491
[manifest] label counts:
 label
gunshot        1670
glass_break     482
alarm           409
water           238
fire_alarm      171
crying          146
scream           94
bicycle          81
dog_bark         40
knock            40
car_horn         40
baby_cry         40
cat_meow         40
Name: count, dtype: int64
[split] trainval=2794 test=697

========== FOLD 1 ==========
train=2236 val=558 test=697
train label counts:
 label
gunshot        1066
glass_break     303
alarm           264
water           159
fire_alarm      111
crying           96
scream           55
bicycle          54
car_horn         27
cat_meow         27
baby_cry         26
dog_bark         24
knock            24
Name: count, dtype: int64
val label counts:
 label
gunshot        268
glass_break     78
alarm           62
water           36
fire_alarm      27
crying          21
scream          21
bicycle         12
dog_bark         9
cat_meow         8
baby_cry      

In [ ]:
import pandas as pd, os
summary_path = f'{RUN_ROOT}/kfold_summary.csv'
summary = pd.read_csv(summary_path)
display(summary)
print('best model:', f'{RUN_ROOT}/best_model.pt')


,fold,best_epoch,checkpoint,val_loss,val_acc,val_macro_f1,val_weighted_f1,test_loss,test_acc,test_macro_f1,test_weighted_f1
0,1,13,/content/drive/MyDrive/edge_audio_runs/effat_m...,0.401766,0.863799,0.831765,0.867574,0.447271,0.846485,0.823091,0.853345
1,2,16,/content/drive/MyDrive/edge_audio_runs/effat_m...,0.427627,0.848485,0.823232,0.854478,0.441567,0.852224,0.844462,0.857692
2,3,15,/content/drive/MyDrive/edge_audio_runs/effat_m...,0.443271,0.851521,0.802587,0.859117,0.451869,0.850789,0.830572,0.857665
3,4,19,/content/drive/MyDrive/edge_audio_runs/effat_m...,0.462529,0.860215,0.801273,0.867482,0.432646,0.847920,0.830097,0.854752
4,5,19,/content/drive/MyDrive/edge_audio_runs/effat_m...,0.400075,0.883513,0.866063,0.888002,0.445671,0.840746,0.822753,0.847240


best model: /content/drive/MyDrive/edge_audio_runs/effat_mn04_kfold/best_model.pt


## 단일 wav 예측 테스트


In [ ]:
import glob, os
candidates = glob.glob(f'{DATA_ROOT}/**/*.wav', recursive=True)
print('num wav:', len(candidates))
AUDIO = candidates[0] if candidates else ''
print('AUDIO =', AUDIO)

if AUDIO:
    !python predict_efficientat.py       --checkpoint "$RUN_ROOT/best_model.pt"       --audio "$AUDIO"       --efficientat-dir /content/EfficientAT       --topk 5
else:
    print('No wav files found under DATA_ROOT')


num wav: 3491
AUDIO = /content/drive/MyDrive/edge_audio_dataset/총/S-210916_W_102_D_205_0001.wav
/usr/local/lib/python3.12/dist-packages/torchvision/ops/misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(
MN(
  (features): Sequential(
    (0): ConvNormActivation(
      (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): ConvNormActivation(
          (0): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=8, bias=False)
          (1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): ConvNormActivation(
          (0): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), bias=Fals